# Central Pb–Pb in a jet-aligned frame

Generate $\sqrt{s_{NN}}=5.02$ TeV Pb–Pb events in the 0–10% bin. Start with 100 generator attempts and select reconstructed jets near 100 GeV. Increase `ATTEMPTS` to 1000 for smoother averages.

Run this notebook with a kernel from an environment with cppyy and `pip install 'hepyy[hepbundle,jewel-cpp]' hepyy-utils numpy matplotlib ipywidgets`. Build `jewel-run-events` and `jewel-medium-probe` from the same jewel-cpp checkout using the [README build instructions](../README.md#build-and-test). The wheel supplies the Python interface and dependencies. This example uses the checkout’s executables for the full event record and the optical-medium query.

The default samples impact parameters from JEWEL’s optical centrality model. Set the `JEWEL_GLAUBER_EVENTS` environment variable to a Glauber CSV to use the validated hard-event Glauber weighting. The transverse medium profile remains optical in either case.

In [ ]:
%matplotlib inline
import json
import math
import os
from pathlib import Path
import subprocess
import sys
import tempfile
import time

from IPython.display import display
import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import PowerNorm
from matplotlib.patches import Circle
import numpy as np

from hepyy_utils.jewel.matching import cluster_final_jets

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
build = Path(os.environ.get("JEWEL_BUILD_DIR", root / "build-pythia8")).expanduser().resolve()
runner, probe = (build / name for name in ("jewel-run-events", "jewel-medium-probe"))
for executable in (runner, probe):
    if not executable.is_file():
        raise FileNotFoundError(f"Build {executable.name} or set JEWEL_BUILD_DIR.")
sys.path.insert(0, str(root / "tools"))
from native_geometry_analysis import associate_geometry

ATTEMPTS = 100
SEED = 2601001
CENTRALITY = (0, 10)
PTHAT_MIN, PTHAT_MAX = 70, 160  # GeV
LEADING_WINDOW = (75, 125)    # GeV, applied to reconstructed jets
SNAPSHOT_TIME = 0.6          # fm/c at z=0
GEOMETRY_STEP = 0.02         # fm/c
glauber_csv = os.environ.get("JEWEL_GLAUBER_EVENTS")

## Generate and match

The higher hard-scale window uses ISR-only, as the native MPI calibration covers 20–100 GeV. Keep recoils, run JEWEL thermal subtraction and exclude thermal residuals and neutrinos from clustering. Cluster anti-$k_t$ $R=0.4$ jets over full phase space, then match the two shower-start legs to distinct jets with $\Delta R\leq0.3$. Apply the jet cuts afterward. The rotation uses the highest-$p_T$ jet within $|\eta|<2$ and points it along $+y'$, even if it has no leg match.

In [ ]:
def rotate_to_north(phi):
    angle = math.pi / 2 - phi
    cos, sin = math.cos(angle), math.sin(angle)
    return np.array([[cos, -sin], [sin, cos]])


def select_event(event):
    final = np.array([p["momentum"] for p in event["subtracted_particles"]
                      if p["final"] and not p["thermal_residual"]
                      and abs(p["pdg"]) not in (12, 14, 16)], dtype=float).reshape(-1, 4)
    jets = cluster_final_jets(*final.T, radius=0.4)
    matches, by_jet = associate_geometry(event, jets, min_pt=30, max_abs_eta=2)
    accepted = [j for j in jets if abs(j["eta"]) < 2 and j["pt"] >= 30]
    if not accepted or not LEADING_WINDOW[0] <= accepted[0]["pt"] < LEADING_WINDOW[1]:
        return None

    rotation = rotate_to_north(accepted[0]["phi"])
    starts = {p["id"]: p for p in event["initial_hard"]}
    # A retained recoil with collision ancestry records the production vertex
    # of that recoil. Read both active and archived records by stable ID.
    recoil_sites = {p["id"]: p["production_fm"][:2]
                    for field in ("particles", "archived")
                    for p in event[field] if p["collision_source"] > 0}
    sites = np.array(list(recoil_sites.values()), dtype=float).reshape(-1, 2)
    thermal = np.array([p["momentum"][:2] for p in event["thermal_references"]],
                       dtype=float).reshape(-1, 2)
    return dict(
        attempt=event["attempt"], b=event["impact_parameter_fm"],
        weight=event["weight"], pt_hat=event["pt_hat"], lifetime=event["event_lifetime_fm"],
        origin=np.array(event["geometry"][0]["production_fm"][:2]),
        rotation=rotation, jets=accepted[:2], by_jet=by_jet,
        legs=[(match, np.array(starts[match["geometry"]["shower_start_id"]]["momentum"]))
              for match in matches],
        sites=sites @ rotation.T, thermal=thermal @ rotation.T,
    )


events = []
with tempfile.TemporaryDirectory(prefix="jewel-geometry-") as work:
    output = Path(work) / "events.jsonl"
    args = [str(runner), str(output), str(ATTEMPTS), str(SEED), "medium",
            "--ecm", "5020", "--pt-min", str(PTHAT_MIN), "--pt-max", str(PTHAT_MAX),
            "--isr-only", "--table-energy", "jewel", "--geometry-step", str(GEOMETRY_STEP),
            "--centrality", *map(str, CENTRALITY), "--pthat-bias", "5",
            "--longitudinal-boost", "on", "--jewel-fragmentation", "--keep-recoils", "on",
            "--subtraction-radius", "0.4"]
    if glauber_csv:
        args += ["--glauber-events", str(Path(glauber_csv).expanduser().resolve())]
    started = time.perf_counter()
    result = subprocess.run(args, cwd=work, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(result.stdout[-3000:] + result.stderr[-3000:])
    generation_seconds = time.perf_counter() - started
    with output.open() as source:
        header = json.loads(next(source))
        assert header["geometry_estimator"]["schema"] == "geometry-v1-native"
        for line in source:
            event = json.loads(line)
            if event["type"] == "event":
                selected = select_event(event)
                if selected is not None:
                    events.append(selected)
            elif event["type"] == "summary":
                summary = event

if not events:
    raise RuntimeError("No leading jets passed. Increase ATTEMPTS or widen LEADING_WINDOW.")
matched = sum(e["jets"][0]["jet_id"] in e["by_jet"] for e in events)
print(f'{summary["accepted"]}/{ATTEMPTS} accepted in {generation_seconds:.1f} s. '
      f'{len(events)} events in the leading jet window, {matched} with a leading hard-leg match.')

## Sample the optical medium and trace the two rays

Query the **same C++ medium model** at $z=0$ and the requested proper time for each event’s $b$. The color field is the ideal-gas energy density inferred from its local temperature, $\epsilon=(\pi^2/30)(16+10.5N_f)T^4/(\hbar c)^3$, with $N_f=3$. The generator samples an interaction medium from this profile. The map is a model slice in GeV/fm³ and carries no per-event jet energy deposition or hydrodynamic response.

The dashed rays extend to the overlap boundary, and the solid intervals satisfy $T>0$ along each **three-dimensional shower-start ray**. Their projected lengths reproduce the exported $L_{\mathrm{hot},T}$. The hard parton’s association to a jet is annotated when available. Scatter markers locate stored recoil production vertices across the full evolution. They are not nucleons or a snapshot at the displayed time. The momentum panel shows the generator’s stored **incoming thermal-reference momenta**, separately from the reconstructed jet collection.

In [ ]:
radius = header["geometry_estimator"]["radius_fm"]
step = header["geometry_estimator"]["step_fm"]
xy_edges = np.linspace(-radius - 1, radius + 1, 89)
xy = (xy_edges[:-1] + xy_edges[1:]) / 2
xx, yy = np.meshgrid(xy, xy)
grid = np.column_stack([xx.ravel(), yy.ravel()])
p_edges = np.linspace(-4, 4, 65)
density_scale = math.pi**2 * (16 + 10.5 * 3) / (30 * 0.19732698**3)


def path_samples(origin, p4, geometry, lifetime):
    v = p4[:3] / p4[3]
    vt = np.linalg.norm(v[:2])
    stop = min(lifetime, geometry["L_geom_T_fm"] / vt) if vt > 0 else lifetime
    edges = np.linspace(0, stop, max(1, math.ceil(stop / step)) + 1)
    times = (edges[:-1] + edges[1:]) / 2
    xyz = times[:, None] * v
    xyz[:, :2] += origin
    return np.column_stack([xyz, times]), edges, v


def query_batch(batch):
    config = f"CONFIG 208 3 1 .4 .59 .17 .17 .54 {float(np.float32(6.2))} 0 2 .45 1"
    slices = []
    with tempfile.TemporaryFile(mode="w+t") as queries, tempfile.TemporaryFile(mode="w+t") as response:
        queries.write(config + "\n")
        for event in batch:
            lab = grid @ event["rotation"]
            probes = [np.column_stack([lab, np.zeros(len(grid)), np.full(len(grid), SNAPSHOT_TIME)])]
            paths = []
            for match, p4 in event["legs"]:
                points, times, velocity = path_samples(event["origin"], p4, match["geometry"], event["lifetime"])
                probes.append(points)
                paths.append((match, times, velocity))
            queries.write(f'EVENTB {event["b"]:.17g}\n')
            for points in probes:
                np.savetxt(queries, points, fmt="POINT %.17g %.17g %.17g %.17g")
            slices.append((lab, paths))
        queries.seek(0)
        result = subprocess.run([str(probe)], stdin=queries, stdout=response, stderr=subprocess.PIPE, text=True)
        if result.returncode:
            raise RuntimeError(result.stderr)
        response.seek(0)
        temperatures = np.loadtxt(response, usecols=0)

    offset = 0
    for event, (lab, paths) in zip(batch, slices):
        stop = offset + len(grid)
        event["density"] = (density_scale * temperatures[offset:stop]**4).reshape(xx.shape).astype("float32")
        offset = stop
        lab_x, lab_y = lab.T
        event["support"] = np.logical_and(
            (lab_x - event["b"]/2)**2 + lab_y**2 <= radius**2,
            (lab_x + event["b"]/2)**2 + lab_y**2 <= radius**2,
        ).reshape(xx.shape)
        event["paths"] = []
        for match, times, v in paths:
            stop = offset + len(times) - 1
            hot = temperatures[offset:stop] > 0
            offset = stop
            lengths = match["geometry"]
            np.testing.assert_allclose(np.diff(times)[hot].sum() * np.linalg.norm(v[:2]),
                                       lengths["L_hot_T_fm"], rtol=1e-9, atol=1e-10)
            np.testing.assert_allclose(np.diff(times)[hot].sum() * np.linalg.norm(v),
                                       lengths["L_hot_fm"], rtol=1e-9, atol=1e-10)
            lab_points = event["origin"] + times[:, None] * v[:2]
            rotated = lab_points @ event["rotation"].T
            segments = np.stack([rotated[:-1], rotated[1:]], axis=1)[hot]
            event["paths"].append((match, segments))
    assert offset == len(temperatures)


for i in range(0, len(events), 8):
    query_batch(events[i:i+8])

## Inspect one event at a time

The star is the common dijet production point. The cyan compass arrow is the leading reconstructed jet, aligned north. The second compass arrow is drawn when a second jet passes acceptance. Compass arrows denote directions only. Solid lines and quoted $L_{\mathrm{hot}}$ values belong to hard legs, with their final-jet matches listed below the map.

In [ ]:
plt.rcParams.update({"font.size": 10, "axes.titleweight": "bold", "figure.facecolor": "white"})
ray_colors = ("#73e0ff", "#ffc970")
extent = (xy_edges[0], xy_edges[-1], xy_edges[0], xy_edges[-1])
vmax = max(float(e["density"].max()) for e in events)
p_area = np.diff(p_edges)[0]**2
xy_area = np.diff(xy_edges)[0]**2


def histogram(points, edges):
    return np.histogram2d(*points.T, bins=(edges, edges))[0].T


def panel(ax, image, edges, cmap, label, norm=None):
    lim = (edges[0], edges[-1], edges[0], edges[-1])
    handle = ax.imshow(image, origin="lower", extent=lim, cmap=cmap, norm=norm,
                       aspect="equal", interpolation="nearest")
    ax.figure.colorbar(handle, ax=ax, label=label, shrink=.76, pad=.025)
    return handle


def plot_event(event_index):
    event = events[event_index]
    fig, (ax, momentum) = plt.subplots(1, 2, figsize=(12.6, 6.0), layout="constrained")
    fig.suptitle(f'Pb–Pb {CENTRALITY[0]}–{CENTRALITY[1]}%  |  Attempt {event["attempt"]}  |  '
                 f'b = {event["b"]:.2f} fm  |  leading jet $p_T$ = {event["jets"][0]["pt"]:.1f} GeV',
                 fontsize=15, fontweight="bold")
    panel(ax, event["density"], xy_edges, "magma", "Optical energy density [GeV/fm³]",
          PowerNorm(.75, vmin=0, vmax=vmax))
    ax.set(title=f'Medium at $\\tau$ = {SNAPSHOT_TIME:g} fm/c', xlabel="$x'$ [fm]", ylabel="$y'$ [fm]")
    collision_centres = np.array([[-event["b"]/2, 0], [event["b"]/2, 0]]) @ event["rotation"].T
    for center in collision_centres:
        ax.add_patch(Circle(center, radius, fill=False, edgecolor="#96a3b3",
                            linewidth=.75, linestyle=":", alpha=.85))
    angles = np.linspace(0, math.tau, 361)
    circle = radius * np.column_stack((np.cos(angles), np.sin(angles)))
    for center, opposite in (collision_centres, collision_centres[::-1]):
        edge = center + circle
        edge[np.linalg.norm(edge - opposite, axis=1) > radius + 1e-9] = np.nan
        ax.plot(*edge.T, color="#d5dee8", linewidth=.75)
    if event["sites"].size:
        ax.scatter(*event["sites"].T, s=15, facecolors="none", edgecolors="white", linewidths=.65,
                   label=f'{len(event["sites"])} recoil production sites')
    origin = event["origin"] @ event["rotation"].T
    labels = []
    jet_colors = {jet["jet_id"]: color for jet, color in zip(event["jets"], ray_colors)}
    for (match, p4), (_, segments) in zip(event["legs"], event["paths"]):
        color = jet_colors.get(match["jet_id"], "#d7d4dc")
        geometry = match["geometry"]
        direction = p4[:2] @ event["rotation"].T
        direction /= np.linalg.norm(direction)
        end = origin + geometry["L_geom_T_fm"] * direction
        ax.plot(*np.stack([origin, end]).T, linestyle="--", color=color, linewidth=1.15, alpha=.8)
        ax.add_collection(LineCollection(segments, colors=[color], linewidths=3, zorder=5))
        jet = f'jet #{match["jet_rank"]}' if match["matched"] else "unmatched"
        if match["ambiguous"]:
            jet += ", ambiguous"
        labels.append(f'Leg {match["leg_id"]} ({jet}): $L_{{hot}}$ = {geometry["L_hot_fm"]:.2f} fm')
    ax.scatter(*origin, marker="*", s=165, zorder=6, color="#ffffff", edgecolor="#21192c",
               linewidth=.9, label="Dijet production point")
    ax.text(.5, -.18, "  |  ".join(labels), ha="center", va="top", transform=ax.transAxes)
    ax.legend(loc="upper left", facecolor="#1b1424", edgecolor="none", labelcolor="white", fontsize=8)
    ax.set(xlim=extent[:2], ylim=extent[2:])

    panel(momentum, histogram(event["thermal"], p_edges) / p_area, p_edges, "viridis",
          "Thermal references per GeV²")
    momentum.set(title=f'Stored thermal momenta ({len(event["thermal"])} references)',
                 xlabel="$p_x'$ [GeV]", ylabel="$p_y'$ [GeV]")
    compass = momentum.inset_axes([.71, .70, .26, .26])
    compass.set(xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), xticks=[], yticks=[])
    compass.set_aspect("equal")
    compass.set_facecolor("#151c2b")
    for jet, color in zip(event["jets"], ray_colors):
        direction = np.array([jet["px"], jet["py"]]) @ event["rotation"].T / jet["pt"]
        compass.annotate("", (.86 * direction), (0, 0), arrowprops=dict(arrowstyle="->", color=color, lw=2))
    compass.set_title("Jet axes", fontsize=8)
    plt.show()


def display_priority(index):
    event = events[index]
    matched_jets = sum(jet["jet_id"] in event["by_jet"] for jet in event["jets"])
    return -matched_jets, abs(event["jets"][0]["pt"] - sum(LEADING_WINDOW)/2)


default_event = min(range(len(events)), key=display_priority)
plot_event(default_event)
if len(events) > 1:
    slider = widgets.IntSlider(min=0, max=len(events)-1, value=default_event, description="Event index",
                               continuous_update=False)
    view = widgets.Output()

    def update_view(change):
        with view:
            view.clear_output(wait=True)
            plot_event(change["new"])

    slider.observe(update_view, names="value")
    display(slider, view)

## Accumulate the jet-aligned events

Use the generator event weights to undo the power-5 hard-scale bias. Average only over events passing the leading-jet selection. The dashed contour encloses positions inside the geometric overlap in at least half of the weighted sample. The second panel shows recoil production counts accumulated over time, with the selected hard vertices overlaid. The momentum histogram is shown in a fixed window around the origin. The length distributions use matched, unambiguous jets only and normalize the sum of their weights independently for each jet rank.

In [ ]:
weights = np.array([e["weight"] for e in events])
mean = lambda images: np.average(images, axis=0, weights=weights)
average_density = mean([e["density"] for e in events])
average_support = mean([e["support"] for e in events])
site_density = mean([histogram(e["sites"], xy_edges) for e in events]) / xy_area
thermal_density = mean([histogram(e["thermal"], p_edges) for e in events]) / p_area
vertices = np.array([e["origin"] @ e["rotation"].T for e in events])
effective_events = weights.sum()**2 / np.sum(weights**2)

fig, axes = plt.subplots(2, 2, figsize=(11.6, 9.5), layout="constrained")
axes = axes.ravel()
fig.suptitle(f'Jet-aligned Pb–Pb {CENTRALITY[0]}–{CENTRALITY[1]}%  |  '
             f'{len(events)} events  |  $N_{{eff}}$ = {effective_events:.1f}',
             fontsize=16, fontweight="bold")
panel(axes[0], average_density, xy_edges, "magma", "GeV/fm³",
      PowerNorm(.75, vmin=0, vmax=vmax))
axes[0].contour(xx, yy, average_support, levels=[.5], colors=["white"], linewidths=[.9], linestyles="--")
axes[0].set(title=f'Mean optical density at $\\tau$ = {SNAPSHOT_TIME:g} fm/c', xlabel="$x'$ [fm]", ylabel="$y'$ [fm]")
panel(axes[1], site_density, xy_edges, "inferno", "Vertices per event per fm²",
      PowerNorm(.5, vmin=0))
axes[1].scatter(*vertices.T, s=10, marker="+", color="#83e7ff", alpha=.65, label="Hard vertices")
axes[1].legend(loc="upper left", fontsize=8)
axes[1].set(title="Recoil production sites", xlabel="$x'$ [fm]", ylabel="$y'$ [fm]")
panel(axes[2], thermal_density, p_edges, "viridis", "Thermal references per event per GeV²",
      PowerNorm(.55, vmin=0))
axes[2].set(title="Incoming thermal momenta", xlabel="$p_x'$ [GeV]", ylabel="$p_y'$ [GeV]")
by_rank = [[], []]
for event in events:
    for rank, jet in enumerate(event["jets"]):
        match = event["by_jet"].get(jet["jet_id"])
        if match and not match["matching_ambiguous"]:
            by_rank[rank].append((match["geometry"]["L_hot_fm"], event["weight"]))
length_edges = np.linspace(0, max((length for rank in by_rank for length, _ in rank), default=1) + .1, 18)
for rank, label, color in zip(by_rank, ("Leading jet", "Second jet"), ("#117ca0", "#b66e11")):
    if rank:
        lengths, selected_weights = np.array(rank).T
        counts = np.histogram(lengths, bins=length_edges, weights=selected_weights)[0]
        axes[3].stairs(counts / counts.sum(), length_edges, label=f"{label} ({len(rank)})", color=color, linewidth=2)
axes[3].set(title="Matched hard-leg lengths", xlabel="$L_{hot}$ [fm]", ylabel="Weighted fraction per bin")
if any(by_rank):
    axes[3].legend(frameon=False)
axes[3].grid(axis="y", alpha=.24)
plt.show()